# Aula 16 — Laboratório: tamanho de efeito e significância prática

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/02-statistics/notebooks/16-tamanho-efeito-laboratorio.ipynb)

Este laboratório separa três dimensões: evidência estatística, magnitude e relevância para uma decisão.

**Dependências:** Python 3.10+, NumPy, pandas, SciPy e Matplotlib.  
**Reprodutibilidade:** semente fixa 20260907.  
**Convenção:** em comparações de perda, valores positivos de A − B favorecem o modelo B.

## Objetivos

Você irá:

1. manter um efeito fixo e observar o p-value cair quando \(n\) cresce;
2. comparar significância estatística com uma SESOI pré-especificada;
3. calcular efeito bruto, relativo, intervalo e \(d_z\) pareado;
4. calcular Cohen \(d\) e Hedges \(g\) para grupos independentes;
5. comparar medidas para proporções;
6. calcular probabilidade de superioridade com empates.

A escala original permanece no centro da interpretação. A padronização é complementar.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from IPython.display import display
from scipy.stats import t, ttest_1samp

SEED = 20260907
ALPHA = 0.05

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seed:", SEED)

## 1. O efeito fica fixo; a evidência muda com \(n\)

No teste t de uma amostra aplicado às diferenças pareadas,

\[
t=d_z\sqrt{n}.
\]

Fixaremos \(d_z=0{,}10\). Isso representa a mesma magnitude padronizada em todos os cenários. O que muda é a precisão.

In [ ]:
dz_fixo = 0.10
tamanhos = np.array([25, 100, 400, 2_500, 10_000])
t_observados = dz_fixo * np.sqrt(tamanhos)
p_values = 2 * t.sf(np.abs(t_observados), df=tamanhos - 1)

efeito_n = pd.DataFrame({
    "n": tamanhos,
    "d_z": dz_fixo,
    "|t|": t_observados,
    "p-value bilateral": p_values,
})
display(efeito_n)

assert np.allclose(efeito_n["d_z"], dz_fixo)
assert p_values[-1] < 1e-20
assert p_values[0] > ALPHA

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(tamanhos, -np.log10(p_values), marker="o", color="#4c78a8", linewidth=2)
ax.axhline(-np.log10(ALPHA), color="#e45756", linestyle="--", label="p = 0,05")
ax.set_xscale("log")
ax.set(xlabel="Tamanho amostral n (escala log)",
       ylabel="−log10(p-value)",
       title="O p-value cai embora o efeito permaneça d_z = 0,10")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Significância e relevância são eixos distintos

Suponha que a diferença bruta positiva represente redução de erro e que a equipe tenha definido \(\Delta_{min}=0{,}50\) como menor efeito relevante.

- Um efeito de 0,60 pode ser relevante, mas ainda impreciso em uma amostra pequena.
- Um efeito de 0,05 pode ser detectado com enorme precisão, mas continuar dez vezes abaixo do limiar.

A decisão não deve ser inferida apenas do p-value.

In [ ]:
cenarios = pd.DataFrame({
    "cenário": ["efeito relevante, amostra pequena", "efeito mínimo, amostra enorme"],
    "efeito bruto": [0.60, 0.05],
    "desvio das diferenças": [1.50, 0.50],
    "n": [20, 10_000],
})
cenarios["d_z"] = cenarios["efeito bruto"] / cenarios["desvio das diferenças"]
cenarios["|t|"] = cenarios["d_z"] * np.sqrt(cenarios["n"])
cenarios["p-value bilateral"] = [
    2 * t.sf(abs(stat), df=int(n) - 1)
    for stat, n in zip(cenarios["|t|"], cenarios["n"])
]
cenarios["supera SESOI 0,50?"] = cenarios["efeito bruto"] >= 0.50
cenarios["p<0,05?"] = cenarios["p-value bilateral"] < ALPHA

display(cenarios)

assert cenarios.loc[0, "supera SESOI 0,50?"] and not cenarios.loc[0, "p<0,05?"]
assert not cenarios.loc[1, "supera SESOI 0,50?"] and cenarios.loc[1, "p<0,05?"]

## 3. Modelos avaliados nos mesmos casos: efeito pareado

Construiremos 300 diferenças com média e desvio-padrão controlados. Isso torna o exemplo didático verificável:

\[
D_i=L_{A,i}-L_{B,i},\qquad
d_z=\frac{\bar D}{s_D}.
\]

O intervalo t para a diferença média preserva a unidade da perda.

In [ ]:
rng = np.random.default_rng(SEED)
n_casos = 300

# Vetor padronizado exatamente para média 0 e desvio amostral 1.
z = rng.normal(size=n_casos)
z = (z - z.mean()) / z.std(ddof=1)

diferencas = 0.041 + 0.141 * z
dificuldade = rng.normal(0.72, 0.10, size=n_casos)
perda_a = dificuldade
perda_b = perda_a - diferencas

media_a = perda_a.mean()
media_b = perda_b.mean()
efeito_bruto = diferencas.mean()
efeito_relativo = efeito_bruto / media_a
sd_d = diferencas.std(ddof=1)
dz = efeito_bruto / sd_d
se = sd_d / np.sqrt(n_casos)
critico = t.ppf(0.975, df=n_casos - 1)
ic = (efeito_bruto - critico * se, efeito_bruto + critico * se)
teste = ttest_1samp(diferencas, popmean=0.0, alternative="two-sided")

resumo_pareado = pd.DataFrame({
    "medida": ["perda média A", "perda média B", "A−B", "redução relativa", "d_z",
               "IC 95% inferior", "IC 95% superior", "p-value"],
    "valor": [media_a, media_b, efeito_bruto, efeito_relativo, dz, ic[0], ic[1], teste.pvalue],
})
display(resumo_pareado.round({"valor": 9}))

assert np.isclose(efeito_bruto, 0.041, atol=1e-12)
assert np.isclose(sd_d, 0.141, atol=1e-12)
assert np.isclose(dz, 0.041 / 0.141, atol=1e-12)
assert 0.024 < ic[0] < 0.026 and 0.056 < ic[1] < 0.058

### Interpretação

A diferença é positiva e o intervalo não inclui zero. Entretanto, para uma SESOI de 0,10 unidade, todo o intervalo permanece abaixo do efeito exigido. A conclusão adequada é “melhoria pequena e precisamente estimada, insuficiente para o critério”, não apenas “resultado significativo”.

In [ ]:
sesoi = 0.10

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.errorbar(efeito_bruto, 0, xerr=[[efeito_bruto - ic[0]], [ic[1] - efeito_bruto]],
            fmt="o", color="#4c78a8", capsize=6, markersize=8, label="efeito e IC 95%")
ax.axvline(0, color="black", linewidth=1.2, label="efeito nulo")
ax.axvline(sesoi, color="#e45756", linestyle="--", linewidth=2, label="SESOI = 0,10")
ax.set(xlabel="Redução média da perda (A − B)", yticks=[],
       title="Precisão estatística não garante relevância prática",
       xlim=(-0.02, 0.13))
ax.legend(loc="upper center", ncol=3)
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Grupos independentes: Cohen \(d\) e Hedges \(g\)

Para dois grupos independentes, padronizamos pela dispersão combinada. Em amostras pequenas, aplicamos

\[
g=Jd,\qquad J\approx1-\frac{3}{4df-1}.
\]

Esta fórmula não deve ser aplicada mecanicamente a dados pareados.

In [ ]:
rng = np.random.default_rng(SEED + 1)
n_a, n_b = 30, 28
grupo_a = rng.normal(loc=10.0, scale=2.0, size=n_a)
grupo_b = rng.normal(loc=9.0, scale=2.2, size=n_b)

media_ga, media_gb = grupo_a.mean(), grupo_b.mean()
sd_a, sd_b = grupo_a.std(ddof=1), grupo_b.std(ddof=1)
df = n_a + n_b - 2
sd_pooled = np.sqrt(((n_a - 1) * sd_a**2 + (n_b - 1) * sd_b**2) / df)
cohen_d = (media_ga - media_gb) / sd_pooled
J = 1 - 3 / (4 * df - 1)
hedges_g = J * cohen_d

resumo_ind = pd.DataFrame({
    "medida": ["média A", "média B", "s_pooled", "Cohen d", "J", "Hedges g"],
    "valor": [media_ga, media_gb, sd_pooled, cohen_d, J, hedges_g],
})
display(resumo_ind.round({"valor": 6}))

assert 0 < J < 1
assert abs(hedges_g) < abs(cohen_d)
assert np.sign(hedges_g) == np.sign(cohen_d)

## 5. Desfecho binário: qual denominador responde à decisão?

Considere 800 sucessos em 1.000 casos com A e 820 em 1.000 com B. Calcularemos:

- diferença de riscos em pontos percentuais;
- ganho relativo sobre sucessos;
- redução relativa do erro;
- razão de riscos;
- razão de chances.

Nenhuma medida deve esconder as taxas de base.

In [ ]:
p_a, p_b = 0.80, 0.82

diferenca_risco = p_b - p_a
ganho_relativo = diferenca_risco / p_a
erro_a, erro_b = 1 - p_a, 1 - p_b
reducao_relativa_erro = (erro_a - erro_b) / erro_a
rr = p_b / p_a
odds_a, odds_b = p_a / (1 - p_a), p_b / (1 - p_b)
oratio = odds_b / odds_a

medidas_binarias = pd.DataFrame({
    "medida": ["taxa A", "taxa B", "diferença absoluta", "ganho relativo",
               "redução relativa do erro", "RR", "OR"],
    "valor": [p_a, p_b, diferenca_risco, ganho_relativo,
              reducao_relativa_erro, rr, oratio],
})
display(medidas_binarias.round({"valor": 6}))

assert np.isclose(diferenca_risco, 0.02)
assert np.isclose(ganho_relativo, 0.025)
assert np.isclose(reducao_relativa_erro, 0.10)
assert not np.isclose(rr, oratio)

## 6. Probabilidade de superioridade

Para duas amostras independentes,

\[
A=P(X>Y)+\frac{1}{2}P(X=Y).
\]

A implementação explícita abaixo conta todas as comparações e metade dos empates. É adequada para o pequeno exemplo didático; bases grandes pedem algoritmos por postos para evitar a matriz completa.

In [ ]:
x = np.array([4, 5, 5, 7, 8])
y = np.array([3, 3, 7, 7])

comparacoes = x[:, None] - y[None, :]
vitorias = np.sum(comparacoes > 0)
empates = np.sum(comparacoes == 0)
total = comparacoes.size
A = (vitorias + 0.5 * empates) / total

print("Vitórias de X:", vitorias)
print("Empates:", empates)
print("Total de pares:", total)
print(f"Probabilidade de superioridade A: {A:.6f}")

assert vitorias == 12
assert empates == 2
assert total == 20
assert np.isclose(A, 0.65)

O resultado \(A=0{,}65\) significa que uma observação aleatória de X supera uma de Y em 65% das comparações, atribuindo metade de cada empate a X. Ele não diz que 65% das pessoas melhoraram e não cria causalidade.

## 7. Mapa de escolha

A medida principal deve estar perto da decisão:

| Situação | Efeito bruto | Complemento |
|---|---|---|
| Latência/custo | diferença em unidade original | razão, quantis |
| Perda pareada | média das diferenças | \(d_z\) |
| Grupos contínuos | diferença de médias | Hedges \(g\) |
| Taxa/acurácia | pontos percentuais | RR e redução do erro |
| Ordinal/assimétrico | mediana ou quantis | probabilidade de superioridade |

Na próxima aula, bootstrap e permutação serão usados quando a distribuição analítica de uma métrica for inconveniente.

In [ ]:
checagens = {
    "efeito fixo em todos os tamanhos": np.allclose(efeito_n["d_z"], dz_fixo),
    "p-value diminui com n": np.all(np.diff(p_values) < 0),
    "efeito pareado exato": np.isclose(efeito_bruto, 0.041, atol=1e-12),
    "IC pareado verificado": 0.024 < ic[0] < 0.026 and 0.056 < ic[1] < 0.058,
    "Hedges g corrige d em direção a zero": abs(hedges_g) < abs(cohen_d),
    "medidas binárias verificadas": np.isclose(reducao_relativa_erro, 0.10),
    "superioridade com empates verificada": np.isclose(A, 0.65),
}

for nome, passou in checagens.items():
    print(f"{'✓' if passou else '✗'} {nome}")

assert all(checagens.values())
print("\nTodas as verificações passaram.")

## Desafios

1. Troque \(d_z\) de 0,10 para 0,05. Qual \(n\) torna o p-value pequeno? A relevância mudou?
2. Altere a SESOI de 0,10 para 0,03 no exemplo pareado e reinterprete o intervalo.
3. Inverta A e B no cálculo de Cohen \(d\). O que muda e o que permanece?
4. Troque as taxas binárias para 0,98 e 0,99. Compare ganho sobre sucessos e redução do erro.
5. Crie amostras X e Y com muitos empates e verifique por que cada empate recebe peso 0,5.
6. Escreva um resultado que inclua efeito bruto, IC, medida padronizada, direção e limiar prático.